# Transcription — Pathway B: Meta MMS

For languages covered by Meta's [Massively Multilingual Speech (MMS)](https://ai.meta.com/blog/multilingual-model-speech-recognition/) `mms-1b-all` model that are not supported by Whisper.

**Run the language check cell (Section 1) before loading the model** — MMS marketing claims 1,100+ languages, but coverage of North American Indigenous languages is uneven in practice.

## MMS components: ASR vs LID vs TTS

The [MMS language coverage page](https://dl.fbaipublicfiles.com/mms/misc/language_coverage_mms.html) lists languages across three separate components:

| Component | What it does | What this notebook needs |
|---|---|---|
| **ASR** | Speech → text (transcription) | **Yes — this is what we use** |
| **LID** | Detects which language is being spoken | No |
| **TTS** | Text → speech | No |

A language appearing under LID (language identification) does **not** mean it can be transcribed. For example, Navajo appears under LID but has no ASR support — MMS can detect that someone is speaking Navajo, but it cannot produce a transcript. Only languages with ASR support work with this notebook.

## Full language list

The file [`../docs/mms_indigenous_languages.tsv`](../docs/mms_indigenous_languages.tsv) contains all 530+ Indigenous languages with confirmed MMS ASR support, organized by region. The language check cell (Section 1) uses this file to show you the language name and region when you enter a code.

## Notable Indigenous languages with MMS ASR support (not in Whisper)

### North America / Arctic (all 8 ASR-supported languages)

| Language | Code | Region | Notes |
|---|---|---|---|
| Cree, Plains | `crk` | North America | MMS also supports `crk-script_syllabics` |
| Ojibwa, Northwestern | `ojb` | North America | MMS also supports `ojb-script_syllabics` |
| Paiute, Northern | `pao` | North America | |
| Tewa | `tew` | North America | |
| Tłı̨chǫ (Dogrib) | `dgr` | North America | |
| Tohono O'odham | `ood` | North America | |
| Gwich'in | `gwi` | North America/Arctic | |
| Yupik, Saint Lawrence Island | `ess` | North America/Arctic | |

### Mesoamerica, South America, and Pacific (selected highlights)

| Language | Code | Region | Notes |
|---|---|---|---|
| K'iche' | `quc` | Mesoamerica | Multiple dialect adapters available |
| Kaqchikel | `cak` | Mesoamerica | Multiple dialect adapters available |
| Mam | `mam` | Mesoamerica | Multiple dialect adapters available |
| Guaraní | `grn` | South America | |
| Quechua (multiple) | `quy`, `quz`, `qub`, `qve`, etc. | South America | 17 Quechua/Quichua varieties |
| Shipibo-Conibo | `shp` | South America | |
| Shuar | `jiv` | South America | |
| Rapa Nui | `rap` | Pacific/Polynesia | |
| Maori | `mri` | Pacific/New Zealand | |

See the full TSV for all 121 Mesoamerican, 125 South American, 105 Pacific/Oceanian, and other regional languages.

**Absent from both Whisper and MMS ASR:** Navajo, Cherokee, Hawaiian, Lakota/Dakota, Mi'kmaq, and many others — use [03c_transcribe_manual.ipynb](03c_transcribe_manual.ipynb) for those.

## A note on MMS training data provenance

For many Indigenous languages, MMS training data was sourced primarily from Bible translations scraped without community consent. This has two practical consequences: output quality may be higher for religious vocabulary than everyday speech, and the model reflects that specific register. Review output carefully with fluent speakers.

All transcription runs locally. No audio leaves your machine.

## 0. Environment Check

```bash
uv sync --extra dev && source .venv/bin/activate
```

The `mms-1b-all` model is ~4 GB and will be downloaded to `~/.cache/huggingface/` on first run.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="tqdm")

issues = []
for pkg, mod in [("transformers", "transformers"), ("torch", "torch"),
                 ("librosa", "librosa"), ("pandas", "pandas"), ("tqdm", "tqdm")]:
    try:
        __import__(mod)
    except Exception as e:
        issues.append(f"{pkg}: {e}")
if issues:
    for i in issues: print(i)
    print("\nRun: uv sync --extra dev")
else:
    print("All dependencies available.")

## 1. Check Language Availability

This cell loads only the processor (lightweight — no 4 GB model download) and checks whether your language code is in the model.

In [ ]:
import csv, os
from transformers import AutoProcessor

MMS_LANGUAGE = "eng"  # Set your language code here

# Load the Indigenous languages TSV for name/region lookup
_tsv_path = os.path.join(os.path.dirname("__file__"), "..", "docs", "mms_indigenous_languages.tsv")
_lang_info = {}
if os.path.exists(_tsv_path):
    with open(_tsv_path, encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            code = row.get("ISO 639-3", "").strip()
            if code and not code.startswith("#"):
                _lang_info[code] = {
                    "name": row.get("Language Name", "").strip(),
                    "region": row.get("Region", "").strip(),
                }

_proc = AutoProcessor.from_pretrained("facebook/mms-1b-all")
available = set(_proc.tokenizer.vocab.keys())

if MMS_LANGUAGE in available:
    info = _lang_info.get(MMS_LANGUAGE.split("-")[0], {})
    if info:
        print(f"✓ '{MMS_LANGUAGE}' is available — {info['name']} ({info['region']})")
    else:
        print(f"✓ '{MMS_LANGUAGE}' is available in MMS ASR.")
    print("Proceed to Section 2.")
else:
    print(f"✗ '{MMS_LANGUAGE}' is NOT in this model.")
    print("Use 03c_transcribe_manual.ipynb instead.")
    base_code = MMS_LANGUAGE.split("-")[0]
    matches = [k for k in sorted(available) if base_code in k]
    if matches:
        print(f"\nPossible related codes: {matches}")
    # Check if it's in the TSV (ASR-confirmed) but not matching the model vocab
    if base_code in _lang_info:
        info = _lang_info[base_code]
        print(f"\nNote: {info['name']} ({base_code}) is in the ASR language list.")
        print("Try variant codes (e.g., with -script_ or -dialect_ suffixes).")

## 2. Configure Paths

In [ ]:
INPUT_DIR  = "../test_data/wavs_export_audacity/"
OUTPUT_CSV = "../test_data/transcripts_mms.csv"

## 3. Load Model

In [ ]:
import torch
from transformers import Wav2Vec2ForCTC, AutoProcessor

model_id = "facebook/mms-1b-all"
print("Loading MMS model (downloads ~4 GB on first run)...")
processor = AutoProcessor.from_pretrained(model_id)
mms_model = Wav2Vec2ForCTC.from_pretrained(model_id)

processor.tokenizer.set_target_lang(MMS_LANGUAGE)
mms_model.load_adapter(MMS_LANGUAGE)
mms_model.eval()
print(f"Language adapter '{MMS_LANGUAGE}' loaded.")

## 4. Transcribe

In [ ]:
import os, csv
import librosa
from tqdm import tqdm

wav_files = sorted(f for f in os.listdir(INPUT_DIR) if f.lower().endswith(".wav"))
results   = []

for filename in tqdm(wav_files, desc="Transcribing (MMS)"):
    file_id  = os.path.splitext(filename)[0]
    filepath = os.path.join(INPUT_DIR, filename)

    waveform, _ = librosa.load(filepath, sr=16000, mono=True)
    inputs = processor(waveform, sampling_rate=16000, return_tensors="pt")

    with torch.no_grad():
        logits = mms_model(**inputs).logits
    predicted_ids = torch.argmax(logits, dim=-1)
    transcript = processor.decode(predicted_ids[0])
    results.append({"file_id": file_id, "transcript": transcript})
    print(f"  {file_id}: {transcript}")

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["file_id", "transcript"])
    writer.writeheader()
    writer.writerows(results)

print(f"Saved to {OUTPUT_CSV}")

## 5. Review and Correct

MMS output requires careful human review. For languages where training data was primarily Bible translations, expect errors in everyday vocabulary, proper nouns, and place names.

Correct the `transcript` column to exactly match what was spoken. Preserve orthography exactly — syllabics characters, diacritics, and special characters are part of the language, not formatting.

Set `knowledge_keeper_reviewed = true` in the metadata after a fluent speaker has confirmed each transcript.

In [ ]:
import pandas as pd
df = pd.read_csv(OUTPUT_CSV)
print(f"{len(df)} transcripts")
df

## Next Steps

- Augment if dataset is too small: [04_augmentation.ipynb](04_augmentation.ipynb)
- Export final dataset: [05_export_ljspeech.ipynb](05_export_ljspeech.ipynb)